In [ ]:
!pip install -q groq pandas sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import time
import random
import threading
import urllib.request
import difflib
from collections import defaultdict

import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/apt_extraction"
os.makedirs(DRIVE_DIR, exist_ok=True)

CANDIDATES_PATH = os.path.join(DRIVE_DIR, "raw_llm_candidates.json")
PROGRESS_PATH   = os.path.join(DRIVE_DIR, "verification_progress.json")
VERIFIED_PATH   = os.path.join(DRIVE_DIR, "groq_verified_full_log.json")
ATTACK_PSEUDO_PATH = os.path.join( DRIVE_DIR,"verified_pseudo_labels.json")

COSINE_PSEUDO_PATH = os.path.join(DRIVE_DIR,"verified_pseudo_labels_cosine.json")

raw_candidates_df = pd.read_json(CANDIDATES_PATH)
print(f"Loaded {len(raw_candidates_df)} candidate triples for verification.")

# from google.colab import files
# uploaded = files.upload()
# raw_candidates_df = pd.read_json(list(uploaded.keys())[0])
# (if you use this instead, also set PROGRESS_PATH/VERIFIED_PATH to plain
#  local filenames rather than DRIVE_DIR paths, since nothing durable is
#  mounted in that case)

Mounted at /content/drive
Loaded 2156 candidate triples for verification.


## MITRE ATT&CK grounding

In [ ]:
def _norm(s: str) -> str:
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")


ATTACK_STIX_URLS = {
    "enterprise": "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json",
    "mobile":     "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/mobile-attack/mobile-attack.json",
    "ics":        "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/ics-attack/ics-attack.json",
}


def load_attack_stix(domains=("enterprise",), cache_dir="attack_stix"):
    os.makedirs(cache_dir, exist_ok=True)
    objects = []
    for domain in domains:
        url = ATTACK_STIX_URLS[domain]
        local_path = os.path.join(cache_dir, f"{domain}-attack.json")
        if not os.path.exists(local_path):
            print(f"Downloading {domain} ATT&CK STIX bundle")
            urllib.request.urlretrieve(url, local_path)
        with open(local_path, "r", encoding="utf-8") as fh:
            bundle = json.load(fh)
        objects.extend(bundle["objects"])
        print(f"  {domain}: {len(bundle['objects'])} STIX objects loaded")
    return objects


ATTACK_TYPE_MAP = {
    "attack-pattern": "attack-pattern",
    "malware":        "malware",
    "tools":          "tool",
    "threat-actor":   "intrusion-set",
    "campaign":       "campaign",
}


def build_attack_kb(stix_objects: list) -> dict:
    kb = defaultdict(dict)
    for obj in stix_objects:
        obj_type = obj.get("type")
        if obj_type not in ATTACK_TYPE_MAP.values():
            continue
        if obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue

        ext_id = next(
            (r["external_id"] for r in obj.get("external_references", [])
             if r.get("source_name") == "mitre-attack"),
            None,
        )
        name = obj.get("name", "")
        names = {name, *obj.get("aliases", []), *obj.get("x_mitre_aliases", [])}
        for n in names:
            if n:
                kb[obj_type][_norm(n)] = (name, ext_id)
    return kb


print("Loading MITRE ATT&CK STIX reference data for verification")
attack_stix_objects = load_attack_stix(domains=("enterprise",))
attack_kb = build_attack_kb(attack_stix_objects)
for _t, _d in attack_kb.items():
    print(f"  {_t}: {len(_d)} known names/aliases")

Loading MITRE ATT&CK STIX reference data for verification
  enterprise: 26086 STIX objects loaded
  campaign: 60 known names/aliases
  intrusion-set: 598 known names/aliases
  malware: 987 known names/aliases
  tool: 114 known names/aliases
  attack-pattern: 672 known names/aliases


In [ ]:
def attack_kb_lookup(name: str, entity_type: str, cutoff: float = 0.85):
    stix_type = ATTACK_TYPE_MAP.get(entity_type)
    if stix_type is None or stix_type not in attack_kb:
        return False, None, None, 0.0

    table = attack_kb[stix_type]
    norm_name = _norm(name)

    if norm_name in table:
        canonical, ext_id = table[norm_name]
        return True, canonical, ext_id, 1.0

    match = difflib.get_close_matches(norm_name, table.keys(), n=1, cutoff=cutoff)
    if match:
        canonical, ext_id = table[match[0]]
        score = difflib.SequenceMatcher(None, norm_name, match[0]).ratio()
        return True, canonical, ext_id, round(score, 3)

    return False, None, None, 0.0


def annotate_with_attack_kb(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    df = df.copy()
    head_matches = df.apply(lambda r: attack_kb_lookup(r["head"], r["head_type"]), axis=1)
    tail_matches = df.apply(lambda r: attack_kb_lookup(r["tail"], r["tail_type"]), axis=1)

    df["head_attack_match"] = [m[0] for m in head_matches]
    df["head_attack_name"] = [m[1] for m in head_matches]
    df["head_attack_id"] = [m[2] for m in head_matches]
    df["tail_attack_match"] = [m[0] for m in tail_matches]
    df["tail_attack_name"] = [m[1] for m in tail_matches]
    df["tail_attack_id"] = [m[2] for m in tail_matches]
    return df


raw_candidates_df = annotate_with_attack_kb(raw_candidates_df)

n_grounded = int(raw_candidates_df["head_attack_match"].sum() + raw_candidates_df["tail_attack_match"].sum()) if not raw_candidates_df.empty else 0
print(f"ATT&CK-grounded entity mentions found in candidates: {n_grounded}")

ATT&CK-grounded entity mentions found in candidates: 544


In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
VERIFIER_MODEL = "openai/gpt-oss-120b"


In [ ]:
import time
import random

MAX_RETRIES = 5

def call_groq(prompt: str, max_tokens: int = 600):
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(2.0)
            return client.chat.completions.create(
                model=VERIFIER_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
                reasoning_effort="low",  # gpt-oss defaults to "medium" if unset,which eats into max_tokens with hidden reasoning before the JSON even starts
            )

        except Exception as e:
            error = str(e)

            # Daily quota check
            if "TPD" in error or "tokens per day" in error.lower():
                print("Groq daily token limit reached. Stopping.")
                return "DAILY_LIMIT_EXCEEDED"

            json_gen_error = (
                "failed to generate json" in error.lower()
                or "failed to validate json" in error.lower()
            )
            retryable = json_gen_error or any(
                code in error for code in ["429", "500", "502", "503", "504"]
            )
            if not retryable or attempt == MAX_RETRIES - 1:
                print(f"Failed call: {error[:150]}")
                return None

            # Exponential backoff.
            wait_time = (2 ** attempt) + random.uniform(0, 0.5)
            reason = "JSON generation failed" if json_gen_error else "Rate limit / server error"
            print(f"{reason}. Retry {attempt + 1}/{MAX_RETRIES} in {wait_time:.1f}s...")
            time.sleep(wait_time)

    return None


In [ ]:
import json

def build_verification_prompt(batch: list) -> str:
    numbered = []
    for i, r in enumerate(batch, 1):
        marked = r.get("marked_text", "")

        numbered.append(
            f'{i}. Marked sentence: "{marked}"\n'
            f'   Claimed head_type: "{r.get("head_type", "")}", '
            f'relation: "{r.get("relation", "")}", '
            f'tail_type: "{r.get("tail_type", "")}"'
        )
    numbered_text = "\n".join(numbered)
    return f"""You are a Cyber Threat Intelligence (CTI) relation verification expert.

Each "Marked sentence" has its head entity wrapped in [E1] ... [/E1] and its
tail entity wrapped in [E2] ... [/E2].

Evaluate each item independently:
1. relation_supported: Is the claimed relation between [E1] and [E2] factually supported by the sentence?
2. span_precise: Do [E1] and [E2] wrap ONLY the entity mention, nothing more or less?

Respond ONLY with a valid JSON array containing exactly {len(batch)} items, in order:
[{{"relation_supported": true/false, "span_precise": true/false, "confidence": 0.0-1.0, "reason": "short explanation"}}]

Items:
{numbered_text}
"""


def parse_verification_batch_response(batch: list, gen_text: str) -> list:
    """Parses JSON batch response into row dicts with verifier results."""
    gen_text = gen_text.strip()

    try:
        start_idx = gen_text.find("[")
        end_idx = gen_text.rfind("]")
        verdicts = json.loads(gen_text[start_idx : end_idx + 1])
    except Exception:
        verdicts = None

    if not isinstance(verdicts, list) or len(verdicts) != len(batch):
        return [
            {
                **r,
                "verifier_relation_supported": False,
                "verifier_span_precise": False,
                "verifier_confidence": 0.0,
                "verifier_reason": f"Parse error: invalid or mismatched batch JSON | raw: {gen_text[:100]}",
                "verifier_parse_failed": True,
            }
            for r in batch
        ]

    rows = []
    for r, v in zip(batch, verdicts):
        try:
            rows.append({
                **r,
                "verifier_relation_supported": bool(v.get("relation_supported", False)),
                "verifier_span_precise": bool(v.get("span_precise", False)),
                "verifier_confidence": float(v.get("confidence", 0.0)),
                "verifier_reason": str(v.get("reason", "")),
                "verifier_parse_failed": False,
            })
        except Exception as e:
            rows.append({
                **r,
                "verifier_relation_supported": False,
                "verifier_span_precise": False,
                "verifier_confidence": 0.0,
                "verifier_reason": f"Item parse error: {e}",
                "verifier_parse_failed": True,
            })

    return rows


In [ ]:
import os
import json
import time
import pandas as pd

SEMANTIC_SIMILARITY_THRESHOLD = 0.80
# Semantic similarity helper
def update_similarity_for_new_rows(df: pd.DataFrame, batch_size: int = 64) -> pd.DataFrame:
    if df.empty:
        return df
    if "semantic_similarity" not in df.columns:
        df["semantic_similarity"] = None
    missing_mask = df["semantic_similarity"].isna()
    if not missing_mask.any():
        return df
    return df

def load_progress(path: str) -> dict:
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"next_index": 0}


def save_progress(progress: dict, path: str):
    with open(path, "w") as f:
        json.dump(progress, f, indent=2)

def append_verified(new_rows: list, output_path: str) -> pd.DataFrame:
    new_df = pd.DataFrame(new_rows)

    if os.path.exists(output_path):
        existing_df = pd.read_json(output_path)
        combined = pd.concat(
            [existing_df, new_df],
            ignore_index=True)
    else:
        combined = new_df

    if not combined.empty:
        combined = combined.drop_duplicates(
            subset=["marked_text", "relation"],
            keep="last")
    combined.to_json(
        output_path,
        orient="records",
        indent=2
    )

    return combined

def save_pseudo_label_checkpoints(verified_df: pd.DataFrame, attack_pseudo_path: str, cosine_pseudo_path: str, verified_path: str, verbose: bool = False,) -> pd.DataFrame:
    if verified_df.empty:
        if verbose:
            print("No verified candidates available for checkpoint.")
        return verified_df

    df = verified_df.copy()
    required_columns = [
        "verifier_relation_supported",
        "verifier_span_precise",
        "verifier_confidence",
        "head_attack_match",
        "tail_attack_match",
    ]

    for col in required_columns:
        if col not in df.columns:
            if col == "verifier_confidence":
                df[col] = 0.0
            else:
                df[col] = False


    is_attack_grounded = (df["head_attack_match"].astype(bool) | df["tail_attack_match"].astype(bool))

    valid_pseudo_labels = df[
        (df["verifier_relation_supported"] == True) & ((df["verifier_confidence"] >= 0.90) | ((df["verifier_confidence"] >= 0.75) & is_attack_grounded ))].copy()

    df = update_similarity_for_new_rows(df, batch_size=64)

    is_semantically_similar = (df["semantic_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD)

    valid_pseudo_labels_cosine = df[(df["verifier_relation_supported"] == True) & ((df["verifier_confidence"] >= 0.90) | ((df["verifier_confidence"] >= 0.75) & is_semantically_similar))].copy()

    df.to_json( verified_path, orient="records",indent=2)

    valid_pseudo_labels.to_json( attack_pseudo_path, orient="records", indent=2)

    valid_pseudo_labels_cosine.to_json( cosine_pseudo_path, orient="records", indent=2)

    if verbose:
        print(
            f"Checkpoint saved:"
            f"\n  Verified candidates : {len(df)}"
            f"\n  ATT&CK pseudo-labels: {len(valid_pseudo_labels)}"
            f"\n  Cosine pseudo-labels : {len(valid_pseudo_labels_cosine)}"
        )

    return df


def run_verification_session(
    df: pd.DataFrame,
    batch_size: int = 8,
    progress_path: str = "verification_progress.json",
    output_path: str = "groq_verified_full_log.json",
    attack_pseudo_path: str = "verified_pseudo_labels.json",
    cosine_pseudo_path: str = "verified_pseudo_labels_cosine.json",
    print_every: int = 100,) -> pd.DataFrame:

    if df.empty:
        print("No candidates to verify.")
        return df

    progress = load_progress(progress_path)
    start_idx = progress.get("next_index", 0)
    records = df.to_dict(orient="records")
    total = len(records)

    if start_idx >= total:
        print(f"All {total} candidates already verified.")
        if os.path.exists(output_path):
            combined = pd.read_json(output_path)
            combined = save_pseudo_label_checkpoints(
                verified_df=combined,
                attack_pseudo_path=attack_pseudo_path,
                cosine_pseudo_path=cosine_pseudo_path,
                verified_path=output_path,
                verbose=True,
            )
            return combined
        return pd.DataFrame()

    print(f"Verifying candidates {start_idx}/{total} (batch_size={batch_size})...")

    start_time = time.time()
    idx = start_idx
    last_printed_count = (start_idx // print_every) * print_every

    while idx < total:
        batch = records[idx : idx + batch_size]
        prompt = build_verification_prompt(batch)
        response = call_groq(prompt, max_tokens=400 * len(batch))

        if response == "DAILY_LIMIT_EXCEEDED":
            print(f"\nStopped at {idx}/{total} — quota reached.")
            print("Run this cell again next session to resume.")
            break

        gen_text = (
            response.choices[0].message.content
            if response
            else ""
        )

        rows = parse_verification_batch_response(batch, gen_text)
        idx += len(batch)

        combined = append_verified(rows, output_path)

        combined = update_similarity_for_new_rows(combined, batch_size=64)

        # Print checkpoint summary every 100 candidates
        should_print = (idx - last_printed_count >= print_every) or (idx >= total)

        combined = save_pseudo_label_checkpoints(
            verified_df=combined,
            attack_pseudo_path=attack_pseudo_path,
            cosine_pseudo_path=cosine_pseudo_path,
            verified_path=output_path,
            verbose=should_print,
        )

        progress["next_index"] = idx
        save_progress(progress, progress_path)

        if should_print:
            print(f"Verified {idx}/{total}")
            last_printed_count = (idx // print_every) * print_every

    elapsed = time.time() - start_time
    print(f"\nCompleted/Paused in {elapsed:.1f}s.")

    combined = (
        pd.read_json(output_path)
        if os.path.exists(output_path)
        else pd.DataFrame()
    )

    if not combined.empty and "verifier_relation_supported" in combined.columns:
        n_relation_ok = int(combined["verifier_relation_supported"].sum())
        n_span_ok = int(combined["verifier_span_precise"].sum())
        print(f"relation_supported = True : {n_relation_ok}/{len(combined)}")
        print(f"span_precise = True       : {n_span_ok}/{len(combined)}")

    return combined

In [ ]:
# Running the verification pipeline on raw LLM candidate triples

verified_df = run_verification_session(
    raw_candidates_df,
    batch_size=8,
    progress_path=PROGRESS_PATH,
    output_path=VERIFIED_PATH,
    attack_pseudo_path=ATTACK_PSEUDO_PATH,
    cosine_pseudo_path=COSINE_PSEUDO_PATH,
)

verified_df.head()

Verifying candidates 32/2156 (batch_size=8)...
Checkpoint saved:
  Verified candidates       : 104
  ATT&CK pseudo-labels     : 53
  Cosine pseudo-labels     : 50
Verified 104/2156
Checkpoint saved:
  Verified candidates       : 200
  ATT&CK pseudo-labels     : 89
  Cosine pseudo-labels     : 83
Verified 200/2156
Checkpoint saved:
  Verified candidates       : 304
  ATT&CK pseudo-labels     : 162
  Cosine pseudo-labels     : 156
Verified 304/2156
Checkpoint saved:
  Verified candidates       : 400
  ATT&CK pseudo-labels     : 220
  Cosine pseudo-labels     : 212
Verified 400/2156
Checkpoint saved:
  Verified candidates       : 504
  ATT&CK pseudo-labels     : 282
  Cosine pseudo-labels     : 274
Verified 504/2156
Checkpoint saved:
  Verified candidates       : 600
  ATT&CK pseudo-labels     : 328
  Cosine pseudo-labels     : 319
Verified 600/2156
Groq daily token limit reached. Stopping.

Stopped at 696/2156 — quota reached.
Run this cell again next session to resume.

Completed/Paused

,marked_text,head_type,tail_type,relation,head,tail,sentence,source_file,head_attack_match,head_attack_name,head_attack_id,tail_attack_match,tail_attack_name,tail_attack_id,verifier_relation_supported,verifier_span_precise,verifier_confidence,verifier_reason,verifier_parse_failed,semantic_similarity
0,related report : https://www.securityartwork.e...,FILEPATH,hash,indicates,Ukraine_election_2019_polls.doc,8a35b6ecdf43f42dbf1e77235d6017faa70d9c68930bdc...,related report : https://www.securityartwork.e...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,False,True,0.72,The sentence only lists a file name and a hash...,False,NaN
1,related report : https://www.securityartwork.e...,FILEPATH,url,communicates-with,Ukraine_election_2019_polls.doc,functiondiscovery[.,related report : https://www.securityartwork.e...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,False,False,0.65,The claimed relation (communicates‑with) is no...,False,NaN
2,]182 [E1] related doc download.doc [/E1] [E2] ...,FILEPATH,hash,indicates,related doc download.doc,8cccdce85beca7b7dc805a7f048fcd1bc8f7614dd7e13c...,]182 related doc download.doc 8cccdce85beca7b7...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,False,True,0.70,The sentence lists a file and a hash without a...,False,NaN
3,]182 [E1] related doc download.doc [/E1] 8cccd...,FILEPATH,url,communicates-with,related doc download.doc,http://beatguitar.com/,]182 related doc download.doc 8cccdce85beca7b7...,apt_subset/APT28__IOC__2019-04-09-ioc-mark.txt,False,None,None,False,None,None,False,True,0.68,No explicit communicates‑with relation between...,False,NaN
4,Internal functions were added to exports (auth...,tools,threat-actor,authored-by,SysWhispers3,the adversary,Internal functions were added to exports (auth...,apt_subset/APT29__CERT.PL__IoC_Reference_.pdf,False,None,None,False,None,None,False,False,0.60,The sentence does not claim that SysWhispers3 ...,False,NaN


In [ ]:
for col in ["verifier_relation_supported", "verifier_span_precise", "verifier_confidence",
            "head_attack_match", "tail_attack_match"]:
    if col not in verified_df.columns:
        verified_df[col] = False if col != "verifier_confidence" else 0.0

is_attack_grounded = ( verified_df["head_attack_match"].astype(bool) | verified_df["tail_attack_match"].astype(bool))

valid_pseudo_labels = verified_df[
    (verified_df["verifier_relation_supported"] == True)
    & (
        (verified_df["verifier_confidence"] >= 0.90)
        | ((verified_df["verifier_confidence"] >= 0.75) & is_attack_grounded)
    )
].copy()

n_span_imprecise_in_accepted = int((~valid_pseudo_labels["verifier_span_precise"]).sum())

print("--- Verification Summary (LLM verifier + ATT&CK grounding) ---")
print(f"Total Candidates Input        : {len(verified_df)}")
print(f"ATT&CK-Grounded Candidates    : {int(is_attack_grounded.sum())}")
print(f"Validated Pseudo-Labels       : {len(valid_pseudo_labels)}")
print(f"Filtered Out (Noise)          : {len(verified_df) - len(valid_pseudo_labels)}")
print(f"  (of which span_precise=False: {n_span_imprecise_in_accepted} -- worth a manual look, "
      f"but not auto-rejected)")

print(
    f"\nCheckpoint file:"
    f"\n{ATTACK_PSEUDO_PATH}"
)
print("\nSaved verified_pseudo_labels.json")

--- Verification Summary (LLM verifier + ATT&CK grounding) ---
Total Candidates Input        : 696
ATT&CK-Grounded Candidates    : 155
Validated Pseudo-Labels       : 386
Filtered Out (Noise)          : 310
  (of which span_precise=False: 21 -- worth a manual look, but not auto-rejected)

Checkpoint file:
/content/drive/MyDrive/apt_extraction/verified_pseudo_labels.json

Saved verified_pseudo_labels.json


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util as st_util

SIMILARITY_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_SIMILARITY_THRESHOLD = 0.5

print(f"Loading similarity model: {SIMILARITY_MODEL_ID}")
similarity_model = SentenceTransformer(SIMILARITY_MODEL_ID)
print("Similarity model loaded.")
print(f"Semantic similarity threshold: {SEMANTIC_SIMILARITY_THRESHOLD}")


def triple_to_statement(head: str, relation: str, tail: str) -> str:
    readable_relation = str(relation).replace("-", " ")
    return f"{head} {readable_relation} {tail}"


def compute_semantic_similarity(
    df: pd.DataFrame, batch_size: int = 64
) -> pd.DataFrame:
    df = df.copy()

    if df.empty:
        df["semantic_similarity"] = pd.Series(dtype=float)
        return df

    if "sentence" in df.columns:
        sentences = df["sentence"].fillna("").astype(str).tolist()
    else:
        sentences = df["marked_text"].fillna("").astype(str).tolist()

    statements = [
        triple_to_statement(h, r, t)
        for h, r, t in zip(df["head"], df["relation"], df["tail"])
    ]

    sent_emb = similarity_model.encode(
        sentences,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=False,
    )

    stmt_emb = similarity_model.encode(
        statements,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=False,
    )

    sims = (st_util.cos_sim(sent_emb, stmt_emb).diagonal().cpu().numpy())

    df["semantic_similarity"] = sims
    return df


def update_similarity_for_new_rows(
    df: pd.DataFrame, batch_size: int = 64
) -> pd.DataFrame:
    df = df.copy()

    if "semantic_similarity" not in df.columns:
        df["semantic_similarity"] = np.nan

    missing_mask = df["semantic_similarity"].isna()

    if not missing_mask.any():
        print("Semantic similarity already exists for all verified candidates.")
        return df

    new_rows = df.loc[missing_mask].copy()
    print(f"Computing semantic similarity for {len(new_rows)} new verified candidates.")

    new_rows = compute_semantic_similarity(new_rows, batch_size=batch_size)

    df.loc[missing_mask, "semantic_similarity"] = new_rows["semantic_similarity"].values

    print("Semantic similarity update complete.")
    return df

Loading similarity model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity model loaded.
Semantic similarity threshold: 0.5


In [ ]:
low_sim = valid_pseudo_labels[valid_pseudo_labels["semantic_similarity"] < SEMANTIC_SIMILARITY_THRESHOLD]

print("--- Semantic Similarity Summary (independent of ATT&CK grounding) ---")
print(f"Accepted pseudo-labels     : {len(valid_pseudo_labels)}")
print(f"Mean semantic similarity   : {valid_pseudo_labels['semantic_similarity'].mean():.3f}")
print(f"Median semantic similarity : {valid_pseudo_labels['semantic_similarity'].median():.3f}")
print(f"Below threshold ({SEMANTIC_SIMILARITY_THRESHOLD})       : {len(low_sim)} "
      f"({100 * len(low_sim) / max(len(valid_pseudo_labels), 1):.1f}%)")

if not low_sim.empty:
    print("\nLowest-similarity accepted triples (worth a manual look):")
    print(low_sim.sort_values("semantic_similarity")
          [["marked_text", "relation", "semantic_similarity"]]
          .head(10).to_string(index=False))

# verified_df.to_json(VERIFIED_PATH, orient="records", indent=2)
# valid_pseudo_labels.to_json("verified_pseudo_labels.json", orient="records", indent=2)
# print(f"\nRe-saved '{VERIFIED_PATH}' and 'verified_pseudo_labels.json' with semantic_similarity added.")

--- Semantic Similarity Summary (independent of ATT&CK grounding) ---
Accepted pseudo-labels     : 386
Mean semantic similarity   : nan
Median semantic similarity : nan
Below threshold (0.5)       : 0 (0.0%)


In [ ]:
is_semantically_similar = verified_df["semantic_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD

valid_pseudo_labels_cosine = verified_df[
    (verified_df["verifier_relation_supported"] == True)
    & (
        (verified_df["verifier_confidence"] >= 0.90)
        | ((verified_df["verifier_confidence"] >= 0.75) & is_semantically_similar)
    )
].copy()

print("--- Verification Summary (LLM verifier + cosine similarity) ---")
print(f"Total Candidates Input          : {len(verified_df)}")
print(f"Semantically-Similar Candidates : {int(is_semantically_similar.sum())}")
print(f"Validated Pseudo-Labels         : {len(valid_pseudo_labels_cosine)}")
print(f"Filtered Out (Noise)            : {len(verified_df) - len(valid_pseudo_labels_cosine)}")

valid_pseudo_labels_cosine.to_json(
    COSINE_PSEUDO_PATH,
    orient="records",
    indent=2
)

print(
    f"\nSaved {COSINE_PSEUDO_PATH}"
)

--- Verification Summary (LLM verifier + cosine similarity) ---
Total Candidates Input          : 696
Semantically-Similar Candidates : 0
Validated Pseudo-Labels         : 377
Filtered Out (Noise)            : 319

Saved /content/drive/MyDrive/apt_extraction/verified_pseudo_labels_cosine.json


In [ ]:
def add_key(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["_key"] = list(zip(df["marked_text"], df["relation"]))
    return df

attack_df = add_key(valid_pseudo_labels)
cosine_df = add_key(valid_pseudo_labels_cosine)

attack_keys = set(attack_df["_key"])
cosine_keys = set(cosine_df["_key"])

both_keys        = attack_keys & cosine_keys
only_attack_keys = attack_keys - cosine_keys
only_cosine_keys = cosine_keys - attack_keys
union_keys       = attack_keys | cosine_keys

print("--- ATT&CK-based vs Cosine-based Acceptance Comparison ---")
print(f"Accepted via ATT&CK pipeline  : {len(attack_keys)}")
print(f"Accepted via cosine pipeline  : {len(cosine_keys)}")
print(f"Accepted by BOTH pipelines    : {len(both_keys)}")
print(f"Only accepted via ATT&CK      : {len(only_attack_keys)}")
print(f"Only accepted via cosine      : {len(only_cosine_keys)}")
if union_keys:
    print(f"Jaccard overlap (both/union)  : {len(both_keys) / len(union_keys):.3f}")

only_attack_df = attack_df[attack_df["_key"].isin(only_attack_keys)]
only_cosine_df = cosine_df[cosine_df["_key"].isin(only_cosine_keys)]

if not only_attack_df.empty:
    print("\nSample triples accepted ONLY via ATT&CK grounding (rejected by cosine):")
    print(only_attack_df[["marked_text", "relation", "semantic_similarity"]]
          .head(10).to_string(index=False))

if not only_cosine_df.empty:
    print("\nSample triples accepted ONLY via cosine similarity (rejected by ATT&CK):")
    print(only_cosine_df[["marked_text", "relation", "head_attack_match", "tail_attack_match"]]
          .head(10).to_string(index=False))

only_attack_df.drop(columns="_key").to_json("disagreement_attack_only.json", orient="records", indent=2)
only_cosine_df.drop(columns="_key").to_json("disagreement_cosine_only.json", orient="records", indent=2)
print("\nSaved disagreement_attack_only.json and disagreement_cosine_only.json for manual review.")

--- ATT&CK-based vs Cosine-based Acceptance Comparison ---
Accepted via ATT&CK pipeline  : 386
Accepted via cosine pipeline  : 377
Accepted by BOTH pipelines    : 377
Only accepted via ATT&CK      : 9
Only accepted via cosine      : 0
Jaccard overlap (both/union)  : 0.977

Sample triples accepted ONLY via ATT&CK grounding (rejected by cosine):
                                                                                                                                                                                                                                                                                        marked_text          relation  semantic_similarity
                                                               ]96:80 00/kernel Hash originally listed on the Mandiant APT41 blog Bash Script 39c8a31dee110 93810c7b142b4 fe8770e8c8d1b 3c09749a2888e cc32d24f4d09 [E1] update.sh [/E1] Downloads [E2] KEYPLUG [/E2] ‘update.so’ from 103.226.155[.         downloads              

In [ ]:
print("\n========== Statistics ==========")
print(f"Total candidates: {len(raw_candidates_df)}")

if not raw_candidates_df.empty and "relation" in raw_candidates_df.columns:
    print(f"Unique relations: {raw_candidates_df['relation'].nunique()}")
    print("\nAccepted relation distribution:")
    print(valid_pseudo_labels["relation"].value_counts() if not valid_pseudo_labels.empty else "None")
    print("\nRejected relation distribution:")
    print(
        verified_df[verified_df["verifier_relation_supported"] == False]["relation"].value_counts()
        if not verified_df.empty else "None"
    )
else:
    print("No valid extracted triples to show statistics for.")


========== Statistics ==========
Total candidates: 2156
Unique relations: 24

Accepted relation distribution:
relation
uses                 100
targets               53
indicates             48
exploits              27
related-to            27
hosts                 25
located-at            21
communicates-with     20
drops                 10
authored-by            8
compromises            7
attributed-to          6
based-on               6
downloads              4
beacons-to             4
duplicate-of           3
originates-from        3
variant-of             3
delivers               2
exfiltrates-to         2
owns                   2
controls               2
consists-of            2
impersonates           1
Name: count, dtype: int64

Rejected relation distribution:
relation
uses                 44
communicates-with    43
hosts                26
targets              25
indicates            23
related-to           16
drops                12
located-at           10
impersonates        

In [ ]:
source_df = valid_pseudo_labels if "valid_pseudo_labels" in globals() and not valid_pseudo_labels.empty else raw_candidates_df

llm_labelled_df = source_df[["marked_text", "head_type", "tail_type", "relation"]].reset_index(drop=True)

print("Relation distribution in converted set:")
print(llm_labelled_df["relation"].value_counts().to_string())

llm_labelled_df.to_json("llm_extracted_labelled_format.json", orient="records", indent=2)
print("\nSaved llm_extracted_labelled_format.json -- same schema as `all_df` in Baseline_Training.ipynb")

Relation distribution in converted set:
relation
uses                 100
targets               53
indicates             48
exploits              27
related-to            27
hosts                 25
located-at            21
communicates-with     20
drops                 10
authored-by            8
compromises            7
attributed-to          6
based-on               6
downloads              4
beacons-to             4
duplicate-of           3
originates-from        3
variant-of             3
delivers               2
exfiltrates-to         2
owns                   2
controls               2
consists-of            2
impersonates           1

Saved llm_extracted_labelled_format.json -- same schema as `all_df` in Baseline_Training.ipynb


In [ ]:
from google.colab import files
import os

files_to_download = [
    ATTACK_PSEUDO_PATH,
    COSINE_PSEUDO_PATH,
    os.path.join(DRIVE_DIR, "llm_extracted_labelled_format.json")
]

for file_path in files_to_download:
    if os.path.exists(file_path):
        files.download(file_path)
    else:
        print(f"File not found, skipped: {file_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File not found, skipped: /content/drive/MyDrive/apt_extraction/llm_extracted_labelled_format.json
